## GPT2

This is my attempt to implement an LLM in a modular manner so that different parts can be used in isolation when needed down the line, and adjustment are self contained. 

### Design

The way I see it there 4 sections to the GPT2 or any other LLMs (at least in the basic form):
1. Pre-Transformer Block
2. Transformer Block
3. Attention Mechanism 
4. Post Transformer Block

- In this implementation the pre-transformer section will be added to the forward method of the `GPT2()` class. 
- The Transformer block will be called `Block()` class containing
  - 3 `LayerNorm`
  - h `attentionHead`
  - 2 `Linear Transformation`
  - 1 `FeedForward`
- The `attentionHead()` will also be a class (experiment).  



In [1]:
import torch 
import torch.nn as nn
import numpy as np

In [4]:
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, q, k, v):
        T = q.shape[-2]
        head_dim = q.shape[-1]
        scores = q @ k.transpose(-1, -2)
        scaled = scores / (head_dim ** 0.5)
        mask = torch.tril(torch.ones(T, T))
        masked = scaled.masked_fill(mask == 0, float=('-inf'))
        w = masked.softmax(dim=-1)
        return w @ v
        

In [6]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_embd = n_embd
        self.n_head = n_head
        self.ln1 = nn.LayerNorm(n_embd)
        self.qkv = nn.Linear(n_embd, 3*n_embd)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.ln2 = nn.LayerNorm(n_embd, n_embd)
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd)
        )
    def review_rehape(self, t):
        B, T, C = t.shape
        t_rev = t.view(B, T, self.n_head, C//self.n_head)
        t_trans = t_rev.transpose(1, 2)
        return t_trans
    
    def concat(self, x):
        B, T, head_dim = x.shape
        x_trans = x.transpose(1, 2)
        x_out = x_trans.contiguous().view(B, T, self.n_head*head_dim)
        return x_out
    
    def forward(self, idx):
        norm_x = self.ln1(idx)
        qkv = self.qkv(norm_x)
        q_raw, k_raw, v_raw = qkv.split(self.n_embd, dim=-1)
        q = self.review_rehape(q_raw)
        k = self.review_rehape(k_raw)
        v = self.review_rehape(v_raw)
        attention_score = Attention(q, k, v)
        x = self.concat(attention_score)
        x_proj = self.out_proj(x) + idx

        x_ln = self.ln2(x_proj)
        x_ff = self.ff(x_ln)
        x_out = x_ff + x_ff
        
        return x_out


**Understandning** `self.blocks = nn.ModuleList([Block(n_embd, n_head) for _ in range(N_t_blocks)])`

- Above we've created a single transformer `Block()` where it must be initialised with the respective dimensions that it'll need throughout the transformer. 
- We then create multiple transformer block using *List Comprehension* ``[expression for varibale in iterable]``.
- We will then use this in the ``forward()``specifically by creating a list in which we append the results, and using the last element in the list. 

```python
block_outputs = [embeddings]
for block in self.blocks:
    block_output.append(block(block_output[-1]))
```

- ``nn.MolduleList(...)`` converts a python list into a Pytorch container so that GD, machine used to compute, parameters etc are kept consistent.

In [ ]:
class GPT2(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, n_head, N_t_blocks):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head) for _ in range(N_t_blocks)])
        self.ln = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx):
        token_emb = self.wte(idx)
        _, T = idx.shape
        pos_emb = self.wpe(torch.arange(T))
        embeddings = token_emb + pos_emb

        # This is wrapped as a list to have something to append to as a standard list
        block_outputs = [embeddings]

        for block in self.blocks:
            block_outputs.append(block(block_outputs[-1]))

        stacked_output = block_outputs[-1]
        normalised_output = self.ln(stacked_output)
        logits = self.lm_head(normalised_output)

        return logits

In [46]:
a = np.ones((2,3,4))
b = np.ones((4,))
print(a)
print(b)
c = a + b
print(c)

[[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]]
[1. 1. 1. 1.]
[[[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]

 [[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]]
